# Procedimento para calcular população de bacias hidrossanitárias
##### Este documento define as etapas para a obtenção do número de habitantes inseridos na Área de Prestação de Serviços de bacias de esgotamento

Importação de bibliotecas:

In [43]:
import pandas as pd
import geopandas as gpd
import os

## 1º Passo: Importação dos dados
##### Setores Censitários: https://www.ibge.gov.br/estatisticas/sociais/trabalho/22827-censo-demografico-2022.html?edicao=41852&t=resultados 
Fazer download da malha de setores centiários por UF
##### Domicílios: https://www.ibge.gov.br/estatisticas/sociais/populacao/38734-cadastro-nacional-de-enderecos-para-fins-estatisticos.html?edicao=38891&t=resultados
Selecionar arquivos por município
##### APS encaminhado pela CORSAN; Bacias delimitadas pelo Analista. Coluna com o nome das bacias também deve ser especificado.

*Buscar código do município em https://www.ibge.gov.br/explica/codigos-dos-municipios.php

In [44]:
#NSR
setores = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Nova Santa Rita\Estudo_domicílios\RS_setores_CD2022.gpkg')
domicilios = pd.read_csv(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Nova Santa Rita\Estudo_domicílios\4313375.csv',delimiter = ';')
aps = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Nova Santa Rita\GPKG\aps.gpkg')
bacias = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Nova Santa Rita\GPKG\bacias.gpkg')
coluna_nome_bacias = 'BACIA_R'
caminho_exportacao = r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Nova Santa Rita\Estudo_domicílios\popdom_bacia_2022-comIgrejas.xlsx'
crs = "EPSG:31982"

## Funções auxiliares

In [45]:
#Funções auxiliares

#contagem de domicílios em cada setor na APS

def somar_extensao_polig(lines_gdf, polys_gdf, poly_id_col="Nome"):
    """
    Retorna um DataFrame com o comprimento (m e km) das linhas dentro de cada polígono.
    Requer que ambos estejam em um CRS projetado em METROS.
    """
    # Garantir colunas necessárias
    polys = polys_gdf[[poly_id_col, "geometry"]].copy()
    lines = lines_gdf[["geometry"]].copy()

    # Corrigir geometrias inválidas (se necessário)
    if hasattr(polys.geometry, "make_valid"):
        polys["geometry"] = polys.geometry.make_valid()
    else:
        polys["geometry"] = polys.buffer(0)

    # Interseção (recorta as linhas por polígono)
    inter = gpd.overlay(lines, polys, how="intersection")

    # Comprimento em metros
    inter["Extensão de Rede (m)"] = inter.geometry.length

    # Soma por polígono
    out = inter.groupby(poly_id_col, as_index=False)["Extensão de Rede (m)"].sum()
    
    return out

def contar_pontos_poligono(polygons, points, polygon_id_col="poly_id", predicate="intersects"):
    if polygons.crs != points.crs:
        points = points.to_crs(polygons.crs)

    # Garante coluna de ID
    if polygon_id_col not in polygons.columns:
        polygons = polygons.reset_index(drop=False).rename(columns={"index": polygon_id_col})

    joined = gpd.sjoin(points, polygons[[polygon_id_col, "geometry"]], predicate=predicate)
    counts = joined.groupby(polygon_id_col).size().rename("n_pontos").reset_index()
    
    result = polygons.merge(counts, on=polygon_id_col, how="left")
    result["n_pontos"] = result["n_pontos"].fillna(0).astype(int)
    
    return result

# Não é necessário mexer nisso abaixo

## 2º Passo: Tratamento dos dados

Conforme Diretriz Corsan (2025), o IBGE considera, para a densidade domiciliar, somente os domicílios particulares ocupados (v0007), o qual não representa a realidade das economias residenciais no cadastro da Corsan/Aegea. Portanto, deve-se recalcular a densidade domiciliar dos setores censitários. Para recalcular a densidade domiciliar, deve-se dividir a população (v0001) pelo total de domicílios particulares (v0003), gerando uma nova coluna “Densidade”.

In [46]:
setores['Densidade'] = setores['v0001']/setores['v0003']

É necessário filtrar os domicílios particulares (COD_ESPECIE = 1) e igrejas (COD_ESPECIE = 8) e transformar csv de domicílios em um arquivo georreferenciado

In [47]:
domparticular = domicilios[(domicilios['COD_ESPECIE'] == 1)|(domicilios['COD_ESPECIE'] == 8)]
domparticular = gpd.GeoDataFrame(domparticular, geometry=gpd.points_from_xy(domparticular.LONGITUDE, domparticular.LATITUDE), crs="EPSG:4326")

Deve-se colocar tudo no mesmo CRS definido

In [48]:
domparticular = domparticular.to_crs(crs)
bacias = bacias.to_crs(crs)
aps = aps.to_crs(crs)
setores = setores.to_crs(crs) #convertendo pra sistema de coordenadas padrão

##### Intersecção entre setores e APS

In [49]:
setores_aps = gpd.clip(setores, aps) #interseção entre setores e APS
domparticular_aps = gpd.clip(domparticular, aps) #interseção entre domicílios e APS

## 3º Passo: Cálculo da população na APS
##### As etapas realizadas são:
- Intersecção entre Setores e APS
- Intersecção entre Domicílios e APS
- Contagem de domicílios em cada setor na APS
- Calculo da população

##### Contagem de domicílios em cada setor na APS

In [50]:
populacao_aps = contar_pontos_poligono(setores_aps, domparticular_aps)

##### Cálculo da população com a densidade e n_pontos criado

In [51]:
populacao_aps['População 2022'] = populacao_aps['Densidade']*populacao_aps['n_pontos']

pop = populacao_aps['População 2022'].sum()
print(f"A população total na APS em 2022 é de {pop:.0f}")
econ = len(domparticular_aps)
print(f"A população total na APS em 2022 é de {econ:.0f}")

A população total na APS em 2022 é de 24587
A população total na APS em 2022 é de 10226


## 4º Passo: Cálculo da população por bacia (2022)

Após delimitar as bacias para pelo menos 90% dos domicílios do IBGE (Censo 2022), deverá ser identificado quantos domicílios estão inseridos em cada bacia. As etapas realizadas são:
- Criação de camada com domicílios classificados por setor e bacia
- Criação de camada com bacias divididas em setores
- Calculo da população com base na densidade de cada domicílio dentro de cada setor dividido pela bacia
- Agrupamento dos valores por bacia, gerando a quantidade de população e domicílios por bacia

##### Intersecções entre domicílios, setores e bacias

In [52]:
camada_unida = gpd.sjoin(
    domparticular_aps,
    bacias, 
    predicate="intersects",
    how="left"
)
camada_unida = camada_unida.drop(columns=['index_right'], errors='ignore')

dompart_setores = gpd.sjoin(
    camada_unida,
    populacao_aps,  
    predicate="intersects",
    how="left"
)

bacias_setores = gpd.sjoin(
    populacao_aps,
    bacias,  
    predicate="intersects",
    how="left"
)

dompart_setores_filtrado = dompart_setores[[coluna_nome_bacias, 'CD_SETOR']]
bacias_setores_filtrado = bacias_setores[[coluna_nome_bacias, 'CD_SETOR','Densidade']]

##### Contagem da quantidade de vezes que uma combinação Setores Censitários + Bacia aparece

In [53]:
# Passo 1: Contar ocorrências de Nome + CD_SETOR na planilha de referência
contagem = (
    dompart_setores_filtrado
    .groupby([coluna_nome_bacias, 'CD_SETOR'])
    .size()
    .reset_index(name='Domicílios')
)

# Passo 2: Fazer merge com o DataFrame base
bacias_setores_filtrado = bacias_setores_filtrado.merge(contagem, on=[coluna_nome_bacias, 'CD_SETOR'], how='left')

# Passo 3: Substituir NaN por 0 (caso não tenha ocorrência)
bacias_setores_filtrado['Domicílios'] = bacias_setores_filtrado['Domicílios'].fillna(0).astype(int)

##### Cálculo da população por combinação Setores Censitários + Bacia

In [54]:
bacias_setores_filtrado['População'] = bacias_setores_filtrado['Domicílios']*bacias_setores_filtrado['Densidade']

##### Soma da população calculada por bacia

In [55]:
bacias_populacao = bacias_setores_filtrado[[coluna_nome_bacias,'Domicílios','População']].groupby(coluna_nome_bacias).sum()
bacias_populacao

,Domicílios,População
BACIA_R,,
BC1,69,156.686325
BC10,11,24.246628
BC11,140,296.235292
BC12,0,0.000000
BC13,183,386.787604
BC14,128,280.137283
BC15,213,471.516009
BC16,118,300.878582
BC17,97,248.740800


##### Exportar excel final

In [56]:
bacias_populacao.to_excel(caminho_exportacao)

# Resultados

In [57]:
pop_aps = populacao_aps['População 2022'].sum()
print(f"A população total na APS em 2022 é de {pop_aps:.0f}")
dom_aps = len(domparticular_aps)
print(f"Os domicílios totais na APS em 2022 é de {dom_aps:.0f}")

resultado_dompop = bacias_populacao.copy()

resultado_dompop['Dom % bacias'] = resultado_dompop['Domicílios']/(resultado_dompop['Domicílios'].sum())
resultado_dompop['Dom % APS'] = resultado_dompop['Domicílios']/dom_aps
resultado_dompop['Pop % bacias'] = resultado_dompop['População']/(resultado_dompop['População'].sum())
resultado_dompop['Pop % APS'] = resultado_dompop['População']/pop_aps

display(resultado_dompop)

A população total na APS em 2022 é de 24587
Os domicílios totais na APS em 2022 é de 10226


,Domicílios,População,Dom % bacias,Dom % APS,Pop % bacias,Pop % APS
BACIA_R,,,,,,
BC1,69,156.686325,0.007238,0.006748,0.006797,0.006373
BC10,11,24.246628,0.001154,0.001076,0.001052,0.000986
BC11,140,296.235292,0.014686,0.013691,0.012851,0.012049
BC12,0,0.000000,0.000000,0.000000,0.000000,0.000000
BC13,183,386.787604,0.019196,0.017896,0.016780,0.015731
BC14,128,280.137283,0.013427,0.012517,0.012153,0.011394
BC15,213,471.516009,0.022343,0.020829,0.020456,0.019178
BC16,118,300.878582,0.012378,0.011539,0.013053,0.012237
BC17,97,248.740800,0.010175,0.009486,0.010791,0.010117


# Extensão e Área das Bacias

In [58]:
eixolog = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Tapes\Aerolevantamento\Fase 7 - Restituicao Planimetrica Digital (MUB)\Shapes\EixoLogradouro.shp')
eixolog =eixolog.to_crs(crs)
bacias_area = bacias.to_crs(crs)

bacias_area_len = somar_extensao_polig(eixolog, bacias_area, poly_id_col="Nome")
bacias_area_len["Área (km²)"] = bacias_area.geometry.area / 10**6

bacias_area_len = bacias_area_len.set_index("Nome")

resultado_final = resultado_dompop.merge(
    bacias_area_len,
    left_index=True,
    right_index=True,
    how="left"
)

resultado_final = resultado_final[['Domicílios','Dom % bacias','Dom % APS','População','Pop % bacias','Pop % APS','Extensão de Rede (m)','Área (km²)']].transpose()

display(resultado_final)

KeyError: "['Nome'] not in index"

In [59]:
resultado_dompop.to_excel(r'C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Nova Santa Rita\Estudo_domicílios\resultadodompop-IGREJAS.xlsx')